<a href="https://colab.research.google.com/github/maclandrol/cours-ia-med/blob/master/05_Analyse_Radiographies_Thoraciques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. Analyse de Radiographies Thoraciques avec l'IA

**Enseignant:** Emmanuel Noutahi, PhD

---

## Objectifs

Dans ce cours pratique, vous allez:
- Charger et visualiser des radiographies thoraciques
- Utiliser un modèle d'IA pré-entraîné pour détecter des pathologies
- Interpréter les résultats de prédiction
- Comprendre les limites et applications cliniques

## Contexte Médical

Les radiographies thoraciques sont l'examen d'imagerie le plus courant en médecine. L'IA peut aider à:
- Détecter des anomalies (pneumonie, cardiomégalie, épanchements, etc.)
- Prioriser les cas urgents
- Assister le radiologue (mais ne le remplace pas!)

**Important:** L'IA est un outil d'aide au diagnostic. La décision finale revient toujours au médecin.

## 1. Installation et Configuration

Nous utilisons **TorchXRayVision**, une bibliothèque spécialisée pour l'analyse de radiographies.

In [ ]:
# Installation des bibliothèques nécessaires
!pip install -q torchxrayvision torch torchvision matplotlib numpy pillow scikit-image opencv-python

print("Installation terminée!")

In [ ]:
# Importation des bibliothèques
import torch
import torchxrayvision as xrv
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Configuration de l'affichage
plt.rcParams['figure.figsize'] = (12, 8)

# Vérification du GPU (optionnel, fonctionne aussi sur CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositif utilisé: {device}")
print(f"Version PyTorch: {torch.__version__}")
print(f"Version TorchXRayVision: {xrv.__version__}")

## 2. Chargement du Modèle d'IA

Nous utilisons un modèle **DenseNet121** pré-entraîné sur des millions de radiographies.
Ce modèle peut détecter 18 pathologies différentes.

In [ ]:
# Chargement du modèle pré-entraîné
print("Chargement du modèle d'IA...")
model = xrv.models.DenseNet(weights="densenet121-res224-all")
model.to(device)
model.eval()  # Mode évaluation (pas d'entraînement)

print("Modèle chargé avec succès!")
print(f"\nPathologies détectables ({len(model.pathologies)}):")
for i, pathology in enumerate(model.pathologies, 1):
    print(f"  {i:2d}. {pathology}")

## 3. Trois Façons de Charger des Images

TorchXRayVision offre plusieurs options pour charger des radiographies.

In [ ]:
# Option 1: Liste des datasets disponibles dans TorchXRayVision
print("=== DATASETS DISPONIBLES DANS TORCHXRAYVISION ===\n")

datasets_disponibles = {
    'NIH': 'NIH ChestX-ray14 (112,120 images, 14 pathologies)',
    'PC': 'PadChest (160,000 images, 174 labels)',
    'CheX': 'CheXpert (224,000 images, 14 pathologies)',
    'MIMIC': 'MIMIC-CXR (377,110 images)',
    'Openi': 'OpenI (3,851 images)',
    'RSNA': 'RSNA Pneumonia Detection Challenge'
}

for nom, description in datasets_disponibles.items():
    print(f"  • {nom:8s}: {description}")

print("\n--- EXEMPLE: Charger un échantillon depuis NIH dataset ---")
print("NOTE: Le téléchargement complet prend du temps. Utilisons un échantillon.")

# Créons plutôt une image simulée réaliste pour la démonstration
def creer_radiographie_simulee(largeur=512, hauteur=512, pathologie=None):
    """
    Crée une radiographie thoracique simulée.
    
    pathologie: None, 'Cardiomegaly', 'Pneumonia', 'Edema', 'Infiltration'
    """
    import cv2
    
    # Image de base (tissus mous)
    img = np.ones((hauteur, largeur), dtype=np.uint8) * 180
    img += np.random.randint(-15, 15, (hauteur, largeur), dtype=np.int16).clip(0, 255).astype(np.uint8)
    
    # Poumons (zones sombres)
    cv2.ellipse(img, (int(largeur*0.35), int(hauteur*0.45)), 
                (int(largeur*0.15), int(hauteur*0.25)), 0, 0, 360, 70, -1)
    cv2.ellipse(img, (int(largeur*0.65), int(hauteur*0.45)), 
                (int(largeur*0.15), int(hauteur*0.25)), 0, 0, 360, 70, -1)
    
    # Médiastin (cœur)
    taille_coeur = 0.12 if pathologie != 'Cardiomegaly' else 0.18
    cv2.ellipse(img, (int(largeur*0.5), int(hauteur*0.55)), 
                (int(largeur*taille_coeur), int(hauteur*0.15)), 0, 0, 360, 140, -1)
    
    # Côtes
    for i in range(6):
        y = int(hauteur * (0.2 + i * 0.08))
        points = np.array([[int(largeur*0.2), y], 
                          [int(largeur*0.4), y + int(hauteur*0.03)],
                          [int(largeur*0.6), y + int(hauteur*0.03)],
                          [int(largeur*0.8), y]], dtype=np.int32)
        cv2.polylines(img, [points], False, 200, 2)
    
    # Ajouter pathologie si spécifiée
    if pathologie == 'Pneumonia' or pathologie == 'Infiltration':
        # Opacité dans poumon droit
        cv2.circle(img, (int(largeur*0.35), int(hauteur*0.40)), int(largeur*0.08), 100, -1)
        cv2.circle(img, (int(largeur*0.33), int(hauteur*0.38)), int(largeur*0.04), 90, -1)
    elif pathologie == 'Edema':
        # Opacités diffuses dans les deux poumons
        for _ in range(8):
            x = np.random.randint(int(largeur*0.25), int(largeur*0.75))
            y = np.random.randint(int(hauteur*0.30), int(hauteur*0.60))
            r = np.random.randint(10, 25)
            cv2.circle(img, (x, y), r, 110, -1)
    
    return img

# Créer une image simulée
img_array = creer_radiographie_simulee(pathologie='Cardiomegaly')
Image.fromarray(img_array).save('demo_xray.jpg')

# Affichage
plt.figure(figsize=(8, 8))
plt.imshow(img_array, cmap='gray')
plt.title("Radiographie Simulée (Cardiomégalie)", fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()

print("\nImage créée: demo_xray.jpg")
print("Type: Radiographie thoracique simulée avec cardiomégalie")
img_path = "demo_xray.jpg"

### Option 2: Upload d'une Image

Vous pouvez télécharger votre propre radiographie.

In [ ]:
# Option 2: Upload depuis votre ordinateur (décommentez pour utiliser)
# from google.colab import files
# uploaded = files.upload()
# if uploaded:
#     img_path = list(uploaded.keys())[0]
#     print(f"Image uploadée: {img_path}")

print("Option 2: Upload - Décommentez le code ci-dessus pour uploader votre image")
print("Pour l'instant, utilisons l'image simulée...\n")

### Option 3: Télécharger depuis une URL

In [ ]:
# Option 3: Télécharger depuis une URL (décommentez pour utiliser)
# !wget -q <VOTRE_URL> -O downloaded.jpg
# img_path = "downloaded.jpg"

print("Option 3: Download - Décommentez et remplacez <VOTRE_URL>")
print("\nUtilisation de l'image simulée pour la suite de la démonstration...")

## 4. Préparation de l'Image pour le Modèle

Le modèle nécessite une préparation spécifique:
- Normalisation avec `xrv.datasets.normalize()` (conversion en plage [-1024, 1024])
- Centrage et redimensionnement à 224x224

In [ ]:
def preparer_image_pour_modele(img_path):
    """
    Prépare une radiographie pour l'analyse par le modèle d'IA.
    Utilise la normalisation standard de TorchXRayVision.
    """
    import skimage.io
    import torchvision.transforms
    
    # 1. Charger l'image
    img = skimage.io.imread(img_path)
    
    # 2. Normaliser avec la fonction de TorchXRayVision
    # Convertit les valeurs pixel en plage [-1024, 1024]
    img = xrv.datasets.normalize(img, 255)
    
    # 3. Convertir en canal unique (si nécessaire)
    if len(img.shape) == 3:
        img = img.mean(2)  # Moyenne des canaux RGB
    img = img[None, ...]  # Ajouter dimension canal
    
    # 4. Appliquer les transformations (centrage et redimensionnement)
    transform = torchvision.transforms.Compose([
        xrv.datasets.XRayCenterCrop(),
        xrv.datasets.XRayResizer(224),
    ])
    
    img = transform(img)
    
    # 5. Convertir en tensor PyTorch
    img_tensor = torch.from_numpy(img)
    img_tensor = img_tensor.to(device)
    
    return img_tensor, img[0]  # Retourner tensor et array pour visualisation

# Préparation de notre image
img_tensor, img_processed = preparer_image_pour_modele(img_path)

# Affichage de l'image traitée
plt.figure(figsize=(6, 6))
plt.imshow(img_processed, cmap='gray')
plt.title("Image Préparée pour le Modèle (224x224)", fontsize=12, fontweight='bold')
plt.axis('off')
plt.show()

print(f"Forme du tensor: {img_tensor.shape}")
print(f"Plage de valeurs: [{img_tensor.min():.2f}, {img_tensor.max():.2f}]")
print("L'image est maintenant prête pour l'analyse!")

## 5. Analyse de la Radiographie

Maintenant, nous utilisons le modèle pour prédire la présence de pathologies.

In [ ]:
# Prédiction avec le modèle
print("Analyse en cours...")

with torch.no_grad():  # Pas besoin de calculer les gradients (mode évaluation)
    predictions = model(img_tensor)
    # Convertir les sorties en probabilités (entre 0 et 1)
    probabilities = torch.sigmoid(predictions).cpu().numpy()[0]

print("Analyse terminée!\n")

# Affichage des résultats
print("=" * 60)
print("RÉSULTATS DE L'ANALYSE")
print("=" * 60)

# Créer un dictionnaire pathologie -> probabilité
resultats = {pathology: prob for pathology, prob in zip(model.pathologies, probabilities)}

# Trier par probabilité décroissante
resultats_tries = sorted(resultats.items(), key=lambda x: x[1], reverse=True)

# Afficher toutes les pathologies
for i, (pathology, prob) in enumerate(resultats_tries, 1):
    # Indicateur visuel
    if prob > 0.5:
        indicateur = "[DÉTECTÉ]"
        couleur = "rouge"
    elif prob > 0.3:
        indicateur = "[SUSPECT]"
        couleur = "orange"
    else:
        indicateur = "          "
        couleur = "vert"
    
    print(f"{i:2d}. {pathology:25s} {prob*100:5.1f}% {indicateur}")

print("=" * 60)

## 6. Visualisation des Résultats

Créons un graphique pour mieux comprendre les résultats.

In [ ]:
# Création d'un graphique à barres
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Graphique 1: Image + Top 5 détections
ax1.imshow(img, cmap='gray')
ax1.set_title("Radiographie Analysée", fontsize=14, fontweight='bold')
ax1.axis('off')

# Graphique 2: Barres des probabilités
pathologies_list = [p for p, _ in resultats_tries]
probs_list = [prob * 100 for _, prob in resultats_tries]

# Couleurs selon le niveau de probabilité
couleurs = ['red' if p > 50 else 'orange' if p > 30 else 'lightblue' for p in probs_list]

bars = ax2.barh(range(len(pathologies_list)), probs_list, color=couleurs, alpha=0.7)
ax2.set_yticks(range(len(pathologies_list)))
ax2.set_yticklabels(pathologies_list, fontsize=9)
ax2.set_xlabel('Probabilité (%)', fontsize=12)
ax2.set_title('Probabilités de Détection par Pathologie', fontsize=14, fontweight='bold')
ax2.axvline(x=50, color='red', linestyle='--', linewidth=1, label='Seuil de détection (50%)')
ax2.axvline(x=30, color='orange', linestyle='--', linewidth=1, label='Seuil de suspicion (30%)')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)
ax2.set_xlim(0, 100)

plt.tight_layout()
plt.show()

## 7. Interprétation Clinique

Comment interpréter ces résultats en tant que professionnel de santé?

In [ ]:
def generer_interpretation(resultats_tries, seuil_haut=0.5, seuil_bas=0.3):
    """
    Génère une interprétation clinique simple des résultats.
    """
    detections_hautes = [(p, prob) for p, prob in resultats_tries if prob > seuil_haut]
    detections_moderees = [(p, prob) for p, prob in resultats_tries if seuil_bas < prob <= seuil_haut]
    
    print("\n" + "="*70)
    print("INTERPRÉTATION CLINIQUE ASSISTÉE PAR IA")
    print("="*70)
    
    if detections_hautes:
        print("\nOK ANOMALIES PROBABLES (> 50%):")
        for pathology, prob in detections_hautes:
            print(f"   • {pathology}: {prob*100:.1f}%")
    
    if detections_moderees:
        print("\nATTENTION ANOMALIES À SURVEILLER (30-50%):")
        for pathology, prob in detections_moderees:
            print(f"   • {pathology}: {prob*100:.1f}%")
    
    if not detections_hautes and not detections_moderees:
        print("\nOK Aucune anomalie majeure détectée")
    
    print("\n" + "="*70 + "\n")

# Génération de l'interprétation
generer_interpretation(resultats_tries)

## 8. Test: Pathologies In-Distribution vs Out-of-Distribution

### In-Distribution (Pathologies que le modèle connaît):
Le modèle a été entraîné sur ces 18 pathologies:

In [ ]:
print("Pathologies DÉTECTABLES (In-Distribution):")
print("="*60)
for i, patho in enumerate(model.pathologies, 1):
    print(f"{i:2d}. {patho}")

print("\n" + "="*60)
print("\nExemples de pathologies OUT-OF-DISTRIBUTION:")
print("  • COVID-19 (n'existait pas lors de l'entraînement)")
print("  • Tuberculose (si non incluse dans l'entraînement)")
print("  • Cancer du poumon spécifique")
print("  • Pathologies pédiatriques rares")
print("\nOBSERVATION: Le modèle tentera de mapper les pathologies OOD")
print("vers les catégories qu'il connaît (ex: COVID → Pneumonia/Infiltration)")

## 9. Impact des Distorsions d'Image

Comment les modifications d'image affectent-elles les prédictions?

In [ ]:
import skimage.transform
import skimage.util

def appliquer_distorsions(img_array):
    """
    Applique diverses distorsions à une image.
    """
    distorsions = {}
    
    # 1. Image originale
    distorsions['Original'] = img_array.copy()
    
    # 2. Rotation
    distorsions['Rotation 15°'] = skimage.transform.rotate(
        img_array, 15, mode='edge', preserve_range=True
    ).astype(np.uint8)
    
    # 3. Zoom (crop central)
    h, w = img_array.shape
    crop_size = int(min(h, w) * 0.7)
    start_h = (h - crop_size) // 2
    start_w = (w - crop_size) // 2
    cropped = img_array[start_h:start_h+crop_size, start_w:start_w+crop_size]
    distorsions['Zoom'] = skimage.transform.resize(
        cropped, img_array.shape, preserve_range=True
    ).astype(np.uint8)
    
    # 4. Bruit
    noisy = skimage.util.random_noise(img_array / 255.0, mode='gaussian', var=0.02)
    distorsions['Bruit'] = (noisy * 255).astype(np.uint8)
    
    # 5. Contraste réduit
    mean_val = img_array.mean()
    low_contrast = ((img_array - mean_val) * 0.5 + mean_val).clip(0, 255)
    distorsions['Faible Contraste'] = low_contrast.astype(np.uint8)
    
    # 6. Sur-exposition
    overexposed = (img_array.astype(float) * 1.3).clip(0, 255)
    distorsions['Sur-exposition'] = overexposed.astype(np.uint8)
    
    return distorsions

# Charger l'image de base
img_base = np.array(Image.open(img_path).convert('L'))

# Appliquer les distorsions
distorsions = appliquer_distorsions(img_base)

print("Distorsions créées:")
for nom in distorsions.keys():
    print(f"  • {nom}")

In [ ]:
# Analyser chaque version avec le modèle
print("Analyse de l'impact des distorsions...\n")

resultats_distorsions = {}

for nom, img_dist in distorsions.items():
    # Sauvegarder temporairement
    temp_path = f'temp_{nom.replace(" ", "_").replace("°", "deg")}.jpg'
    Image.fromarray(img_dist).save(temp_path)
    
    # Préparer et analyser
    try:
        img_tensor_dist, _ = preparer_image_pour_modele(temp_path)
        
        with torch.no_grad():
            pred = model(img_tensor_dist[None,...])
            probs = torch.sigmoid(pred).cpu().numpy()[0]
        
        # Garder top 3 prédictions
        top_indices = np.argsort(probs)[-3:][::-1]
        top_pathologies = [(model.pathologies[i], probs[i]) for i in top_indices]
        
        resultats_distorsions[nom] = {
            'image': img_dist,
            'predictions': top_pathologies
        }
    except Exception as e:
        print(f"Erreur avec {nom}: {e}")
        resultats_distorsions[nom] = {
            'image': img_dist,
            'predictions': [('Erreur', 0.0)]
        }

print("Analyse terminée!")

In [ ]:
# Visualisation comparative
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (nom, data) in enumerate(resultats_distorsions.items()):
    if idx >= 6:
        break
    
    # Afficher l'image
    axes[idx].imshow(data['image'], cmap='gray')
    
    # Titre avec top prédiction
    top_pred = data['predictions'][0]
    titre = f"{nom}\n{top_pred[0]}: {top_pred[1]*100:.1f}%"
    axes[idx].set_title(titre, fontsize=11, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

# Tableau comparatif
print("\n" + "="*80)
print("IMPACT DES DISTORSIONS SUR LES PRÉDICTIONS")
print("="*80)

for nom, data in resultats_distorsions.items():
    print(f"\n{nom:20s}")
    for i, (patho, prob) in enumerate(data['predictions'][:3], 1):
        print(f"  {i}. {patho:25s} {prob*100:5.1f}%")

print("\n" + "="*80)
print("OBSERVATIONS:")
print("  • Rotation légère: Impact minimal sur les prédictions")
print("  • Bruit: Peut réduire la confiance mais garde les mêmes prédictions")
print("  • Contraste/Exposition: Impact important - attention à la qualité!")
print("  • Zoom: Peut manquer des détails périphériques importants")
print("="*80)

## 8. Exercice Pratique: Analyser Votre Propre Image

Téléchargez une radiographie thoracique et analysez-la avec le modèle.

In [ ]:
# Option 1: Télécharger depuis une URL
# Décommentez et modifiez l'URL ci-dessous
# !wget -q <VOTRE_URL_ICI> -O ma_radio.jpg
# mon_image = "ma_radio.jpg"

# Option 2: Upload depuis Google Colab
from google.colab import files
uploaded = files.upload()
if uploaded:
    mon_image = list(uploaded.keys())[0]
    print(f"Image téléchargée: {mon_image}")
    
    # Analyse de votre image
    img_tensor, img_processed = preparer_image_pour_modele(mon_image)
    
    with torch.no_grad():
        predictions = model(img_tensor)
        probabilities = torch.sigmoid(predictions).cpu().numpy()[0]
    
    resultats = {pathology: prob for pathology, prob in zip(model.pathologies, probabilities)}
    resultats_tries = sorted(resultats.items(), key=lambda x: x[1], reverse=True)
    
    # Affichage
    img_display = Image.open(mon_image)
    plt.figure(figsize=(8, 8))
    plt.imshow(img_display, cmap='gray')
    plt.title("Votre Radiographie", fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.show()
    
    generer_interpretation(resultats_tries)
else:
    print("Aucune image téléchargée.")

## 9. Résumé et Points Clés

### Ce que vous avez appris:

1. **Chargement de modèles pré-entraînés**: Utiliser des modèles d'IA sans avoir à les créer
2. **Préparation d'images médicales**: Normalisation et formatage pour l'IA
3. **Interprétation de résultats**: Comprendre les probabilités et seuils de décision
4. **Limites de l'IA**: L'IA aide mais ne remplace pas le jugement clinique

### Applications pratiques:

- **Triage automatique**: Prioriser les cas urgents dans un service d'urgence
- **Double lecture**: Assister les radiologues pour éviter les oublis
- **Télémédecine**: Pré-analyse dans les zones avec peu de radiologues
- **Recherche**: Analyse de grandes cohortes de patients

### Limites importantes:

1. **Qualité de l'image**: Le modèle est sensible à la qualité de la radiographie
2. **Biais de données**: Entraîné sur certaines populations, peut être moins performant sur d'autres
3. **Pathologies rares**: Moins fiable pour des conditions peu représentées dans les données d'entraînement
4. **Contexte clinique**: Ne connaît pas l'histoire du patient, les symptômes, etc.

### Prochaines étapes:

- Cours 6: Segmentation d'images médicales (délimiter les organes/lésions)
- Cours 7: Éthique et déploiement responsable de l'IA en médecine

## 10. Questions de Réflexion

1. **Seuils de décision**: Pourquoi avons-nous choisi 50% comme seuil de détection? Que se passe-t-il si on le baisse à 30% ou le monte à 70%?

2. **Faux positifs vs faux négatifs**: Dans le contexte médical, qu'est-ce qui est plus grave? Un faux positif (alarme injustifiée) ou un faux négatif (pathologie manquée)?

3. **Responsabilité**: Si l'IA manque une pathologie qu'un radiologue aurait détectée, qui est responsable?

4. **Généralisation**: Ce modèle a été entraîné sur des données américaines. Sera-t-il aussi performant sur des patients africains, asiatiques, pédiatriques?

5. **Utilité clinique**: Dans quelles situations cette IA serait-elle la plus utile dans votre pratique?

## 10. Points Clés

### Ce que vous avez appris:
- Utiliser un modèle d'IA pré-entraîné pour analyser des radiographies
- Interpréter les probabilités de détection
- Comprendre les limites: qualité d'image, biais, pathologies hors distribution

### Applications:
- Triage automatique aux urgences
- Assistance au radiologue (double lecture)
- Télémédecine dans zones sous-équipées

### Limites:
- ATTENTION L'IA est un outil d'aide, pas un diagnostic
- ATTENTION Sensible à la qualité d'image et population d'entraînement
- ATTENTION Ne détecte que les pathologies connues (données d'entraînement)